# Evaluating counterfactuals

`nife.counterfactual.evaluate_counterfactuals` measures counterfactuals found by **any** method, NIFE's or not. Every counterfactual explainer has the same evaluation as `evaluate`, with its own model, data and options. What NIFE calls a counterfactual is described in [counterfactual_explanations.ipynb](counterfactual_explanations.ipynb).

Requirements: `pip install "nife[notebooks]"`.

In [1]:
import pandas as pd
from sklearn.datasets import load_wine
from sklearn.ensemble import RandomForestClassifier

from nife.counterfactual import TreeCounterfactualExplainer

X, y = load_wine(return_X_y=True, as_frame=True)
model = RandomForestClassifier(random_state=0).fit(X, y)
explainer = TreeCounterfactualExplainer(model, X, random_state=0)

instances = X.sample(20, random_state=0)
counterfactuals = explainer.explain_many(instances)
explainer.evaluate(instances, counterfactuals)

coverage             0.900000
count               11.833333
validity             1.000000
proximity            0.058231
changed_features     2.111111
redundancy           0.000000
implausibility       0.057939
ynn                  0.225614
diversity            0.045147
dtype: float64

The evaluation takes the instances and their counterfactuals: a DataFrame with the features and an `instance` column, the position of each counterfactual's instance in `instances`, which is the format that `explain_many` returns. Other columns are ignored.

Each metric is computed per counterfactual or per instance, averaged per instance, and then averaged over the instances that have at least one counterfactual; `coverage` and `count` take every instance. Distances are Gower distances (Gower, 1971), so numeric and categorical features can be mixed.

| Metric | Measures | Better | Reference |
|---|---|---|---|
| `coverage` | fraction of the instances with at least one counterfactual | higher | Pawelczyk et al. (2021) |
| `count` | counterfactuals per instance | — | |
| `validity` | fraction of the counterfactuals that reach a counterfactual outcome | higher | Wachter et al. (2018); Mothilal et al. (2020) |
| `proximity` | distance between the instance and its counterfactuals | lower | Mothilal et al. (2020) |
| `changed_features` | number of features changed (sparsity) | lower | Mothilal et al. (2020) |
| `redundancy` | changed features that could be reverted without losing validity | lower | Pawelczyk et al. (2021) |
| `implausibility` | distance to the closest example of the data | lower | Guidotti (2022) |
| `ynn` | fraction of the k nearest examples that also get a counterfactual outcome | higher | Pawelczyk et al. (2021) |
| `diversity` | distance between the counterfactuals of the same instance | higher | Mothilal et al. (2020) |
| `constraint_violations` | fraction of the counterfactuals that change an immutable feature | lower | Pawelczyk et al. (2021) |

## Reading the metrics

Compare the counterfactuals with the evidence of the counterfactual rule of the surrogate tree method: the real wines on the other side of the rule that the model puts in another class (see [TreeCounterfactualExplainer.ipynb](TreeCounterfactualExplainer.ipynb)). The evidence would make poor counterfactuals. It is real, so its `implausibility` is 0, but it changes almost every feature, most of them needlessly (`redundancy`). The counterfactuals change about two features, none of them needlessly:

In [2]:
counterfactuals, explanations = explainer.explain_many(instances, return_explanations=True)
evidence = pd.concat(
    [e.evidence[X.columns].assign(instance=i) for i, e in enumerate(explanations) if len(e.evidence)],
    ignore_index=True,
)
pd.DataFrame({
    "counterfactuals": explainer.evaluate(instances, counterfactuals),
    "evidence of the rule": explainer.evaluate(instances, evidence),
})

,counterfactuals,evidence of the rule
coverage,0.900000,0.900000
count,11.833333,38.222222
validity,1.000000,1.000000
proximity,0.058231,0.273078
changed_features,2.111111,12.790920
redundancy,0.000000,12.720352
implausibility,0.057939,0.000000
ynn,0.225614,0.947124
diversity,0.045147,0.173484


## Options

- `metrics=[...]` computes only some of the metrics.
- `per_instance=True` returns one row per instance instead of the averages.
- `desired`, `immutable_features` and `tolerance` define the counterfactual outcome and the immutable features, as for the explainers; `k` is the number of neighbors of `ynn`.

In [3]:
from nife.counterfactual import evaluate_counterfactuals

evaluate_counterfactuals(instances, counterfactuals, model, X, metrics=["validity", "proximity"])

coverage      0.900000
count        11.833333
validity      1.000000
proximity     0.058231
dtype: float64

In [4]:
explainer.evaluate(instances, counterfactuals, per_instance=True).head()

,counterfactuals,validity,proximity,changed_features,redundancy,implausibility,ynn,diversity
instance,,,,,,,,
0,8,1.0,0.041867,2.0,0.0,0.041867,0.025000,0.042536
1,14,1.0,0.076077,2.0,0.0,0.074513,0.014286,0.056919
2,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,18,1.0,0.049100,2.0,0.0,0.049100,0.100000,0.042547
4,33,1.0,0.081171,3.0,0.0,0.081171,0.715152,0.051007


## New metrics

A new metric is a function of an `EvaluationContext`, registered once with `register_metric` and reported by every evaluation from then on. The context holds the instances, the counterfactuals, their points for the Gower distance and the model's predictions; the docstring of `EvaluationContext` lists them all.

In [5]:
from nife.counterfactual import register_metric


@register_metric("max_change", better="lower", reference="—")
def max_change(ctx):
    """Largest scaled change of a single feature."""
    numeric_cf, numeric_x = ctx.cf_points[0], ctx.x_points[0][ctx.owner]
    return abs(numeric_cf - numeric_x).max(axis=1)


explainer.evaluate(instances, counterfactuals, metrics=["max_change"])

coverage       0.900000
count         11.833333
max_change     0.460493
dtype: float64

A metric returns one value per counterfactual (`per="counterfactual"`, the default) or one per instance (`per="instance"`), NaN where it is not defined.

## References

- Gower, J. C. (1971). A general coefficient of similarity and some of its properties. *Biometrics*, 27(4), 857–871.
- Guidotti, R. (2022). Counterfactual explanations and how to find them: literature review and benchmarking. *Data Mining and Knowledge Discovery*. https://doi.org/10.1007/s10618-022-00831-6
- Mothilal, R. K., Sharma, A., & Tan, C. (2020). Explaining machine learning classifiers through diverse counterfactual explanations. *FAT\* '20*. https://doi.org/10.1145/3351095.3372850
- Pawelczyk, M., Bielawski, S., van den Heuvel, J., Richter, T., & Kasneci, G. (2021). CARLA: A Python library to benchmark algorithmic recourse and counterfactual explanation algorithms. *NeurIPS Datasets and Benchmarks Track*.
- Wachter, S., Mittelstadt, B., & Russell, C. (2018). Counterfactual explanations without opening the black box: automated decisions and the GDPR. *Harvard Journal of Law & Technology*, 31(2), 841–887.